In [ ]:
!pip install diffusers transformers accelerate peft datasets imageio imageio-ffmpeg

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)

Device: cuda


In [ ]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=dtype
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe = pipe.to(device)
# pipe.enable_model_cpu_offload() # Removed to prevent device mismatch during training

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


In [ ]:
!pip install --upgrade torchao

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=2,              # smaller = more stable
    lora_alpha=8,
    target_modules=["to_q", "to_k"],
    lora_dropout=0.05,
    bias="none"
)
pipe.unet = get_peft_model(pipe.unet, lora_config)

In [ ]:
dataset = [
    "A dog running in a green park",
    "A car moving on road",
    "A person walking on beach"
]

In [ ]:
import torch.optim as optim
torch.cuda.empty_cache()

optimizer = optim.Adam(pipe.unet.parameters(), lr=1e-4)

for epoch in range(2):   # 🔥 reduce epochs
    for prompt in dataset:

        latents = torch.randn((1, 4, 4, 32, 32), device=device, dtype=dtype)

        timestep = torch.randint(0, 1000, (1,), device=device).long()

        noise = torch.randn_like(latents, dtype=dtype)
        noisy_latents = latents + noise

        text_input = pipe.tokenizer(prompt, return_tensors="pt").to(device)
        encoder_hidden_states = pipe.text_encoder(**text_input)[0]
        encoder_hidden_states = encoder_hidden_states.to(dtype)

        noise_pred = pipe.unet(
            noisy_latents,
            timestep,
            encoder_hidden_states=encoder_hidden_states
        ).sample

        loss = ((noise_pred - noise) ** 2).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item()}")

Epoch 0 Loss: 0.63623046875
Epoch 1 Loss: 0.64013671875


In [ ]:
prompt = "A cinematic video of a dog running in a green park, realistic motion, detailed fur, soft lighting, high detail, smooth motion, dynamic camera"

result = pipe(
    prompt,
    negative_prompt = "blurry, unclear, low quality, pixelated, unrealistic, artifacts, poor lighting, distorted, bad frame composition",
    num_inference_steps=50,
    num_frames=16,
    guidance_scale=10
)

video_frames = result.frames

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import numpy as np
import imageio

frames = video_frames[0]
frames = [(frame * 255).astype(np.uint8) for frame in frames]

imageio.mimsave("finetuned_video.mp4", frames, fps=4)

print("✅ Video saved as finetuned_video.mp4")

✅ Video saved as finetuned_video.mp4
